In [0]:
SELECT *, ai_query(
 'databricks-meta-llama-3-3-70b-instruct',
 "Can you tell me the name of the US state that serves the provided ZIP code? zip code: " || pickup_zip
) AS response
FROM samples.nyctaxi.trips
LIMIT 10;


WITH aggregated AS (
    SELECT 
        DATE(TPEP_PICKUP_DATETIME) AS ds, 
        SUM(FARE_AMOUNT) AS revenue
    FROM SAMPLES.NYCTAXI.TRIPS
    GROUP BY 1
)
SELECT * FROM AI_FORECAST(
    TABLE(aggregated), 
    horizon => '2016-03-31', -- End date for the forecast
    time_col => 'ds', 
    value_col => 'revenue'
);


--  working with pdfs
-- Assuming your PDFs are in a Unity Catalog volume
-- Step 1: Read files into a table as binary
CREATE OR REPLACE TABLE ai2605.ai.pdf_raw_data AS
SELECT * FROM read_files('/Volumes/ai2605/ai/unstructured/pdfs/', format => 'binaryFile');

-- Step 2: Parse the PDF content
CREATE OR REPLACE TABLE ai2605.ai.parsed_documents AS
SELECT 
    path, 
    ai_parse_document(content) AS parsed_json
FROM ai2605.ai.pdf_raw_data;

-- Step 3: Extract content (e.g., pulling out elements)
SELECT path, element.type, element.content 
FROM ai2605.ai.parsed_documents
LATERAL VIEW EXPLODE(parsed_json:document:elements::array<struct<page_id:bigint,confidence:decimal(4,4),content:string,description:string,id:bigint,type:string>>) AS element;


     